# 2.1-Metric Summary

**Inés Sentís**

Date of execution

In [ ]:
Sys.Date()

## Libraries

In [ ]:
suppressMessages(suppressWarnings({
library(tidyverse)
library(gt)
library(ggpubr)
library(ggrepel)
library(here)
library(glue)
library(readr)
library(dplyr)
library(IRdisplay)
library(ggplot2)
}))

## Parameters

In [ ]:
args <- commandArgs(TRUE)
subproj <- args[2]
source(here(glue("{subproj}/st_analysis/misc/paths.R")))

"{qc}/{plt_dir}" %>%
  glue::glue() %>%
  here::here() %>%
  dir.create(path = .,
           showWarnings = FALSE,
           recursive = TRUE)

"{qc}/{robj_dir}" %>%
  glue::glue() %>%
  here::here() %>%
  dir.create(path = .,
           showWarnings = FALSE,
           recursive = TRUE)

## Load Data

We have 4 different samples, 1 for each capture area from a visium slides.

In [ ]:
id_sp_df <- read_csv(file = here(glue("{spaceranger}/data/sample_id.csv")),show_col_types = FALSE)

In [ ]:
qcmetric_df <- lapply(id_sp_df$gem_id, function(id) {
  tmp_df <- read_csv(file = here(glue("{spaceranger}/results/{id}/outs/metrics_summary.csv")),show_col_types = FALSE)
  tmp_df <- tmp_df %>%
    mutate(sample_id = id_sp_df[id_sp_df$gem_id == id, ] %>% pull(patient)) %>%
    rename(gem_id = `Sample ID`)
  
  return(tmp_df)
}) %>% bind_rows()

In [ ]:
colnames(qcmetric_df) <- stringr::str_replace_all(string = colnames(qcmetric_df),
                                                  pattern = " ",
                                                  replacement = "_")

DT::datatable(qcmetric_df, fillContainer = T) %>%
    DT::formatStyle(
        column = colnames(qcmetric_df),
        backgroundColor = "white"
  )

## Gene Expression QC

We will start by showing the three most relevant metrics (number of reads, estimated number of spots under tissue, fraction of reads in spots under tissue, mean reads per spot, fraction of reads mapped to exonic reads, and median genes per spot) obtained by spaceranger for each of the working libraries. This information will give us an idea of the quality of the experiment as well as the sequencing and the mapping steps.

In [ ]:
top_metrics_gex <- qcmetric_df[, c("gem_id",
                                   "sample_id",
                                   "Number_of_Reads", 
                                   "Number_of_Spots_Under_Tissue", 
                                   "Fraction_Reads_in_Spots_Under_Tissue",
                                   "Mean_Reads_per_Spot",
                                   "Reads_Mapped_Confidently_to_Exonic_Regions",
                                   "Median_Genes_per_Spot")]

In [ ]:
tbl <- top_metrics_gex %>%
  gt() %>%
  fmt_percent(columns = c("Fraction_Reads_in_Spots_Under_Tissue", 
                          "Reads_Mapped_Confidently_to_Exonic_Regions"), 
              scale_values = TRUE, decimals = 2) %>%
  fmt_number(columns = "Number_of_Reads", 
             scale_by = 1 / 1E6, 
             pattern = "{x}M") %>% 
  tab_header(
    title = md("**GEX QC metrics**"),
    subtitle = "spaceranger v1.1.0"
  ) %>%
  cols_label(
    gem_id = md("**GEM ID**"),
    sample_id = md("**SAMPLE ID**"),
    Number_of_Reads = md("**Number of Reads**"),
    Number_of_Spots_Under_Tissue = md("**Number of Spots Under Tissue**"),
    Fraction_Reads_in_Spots_Under_Tissue = md("**Fraction Reads in Spots Under Tissue**"),
    Mean_Reads_per_Spot = md("**Mean Reads per Spot**"),
    Reads_Mapped_Confidently_to_Exonic_Regions = md("**Fraction of Reads Mapped to Exonic Reads**"),
    Median_Genes_per_Spot = md("**Median Genes per Spot**")
  )

# Render with full CSS/HTML
IRdisplay::display_html(gt::as_raw_html(tbl))

## Sequencing QC

First, we will assess the quality of the sequenced libraries before the mapping step. To so do, we will use the "Q30" variables in our dataset, as these report the fraction of the bases with a Q-score >30 for different sequences (barcodes, reads and UMIs).
Q-score is calculated in the following way:
$$Q = -10\log10(p)$$
where *p* is the probability of the base being wrongly called. Thus, bases with a high Q-score are reliable.

In [ ]:
options(repr.plot.width = 15, repr.plot.height =5, verbose = FALSE)
qc_seq_vars <- c("Q30_Bases_in_Barcode",
                 "Q30_Bases_in_RNA_Read",
                 "Q30_Bases_in_UMI")

gg_qc_seq <- purrr::map(qc_seq_vars, function(var) {
  ggplot(qcmetric_df, aes_string(x = "sample_id", y = var, fill = "sample_id")) +
    geom_col() +
    theme_bw() +
    scale_fill_brewer(palette = "Set2") +
    ylim(0, 1) +
    labs(x = "Libraries (SAMPLE IDs)",
         y = stringr::str_c(stringr::str_replace_all(var, "_", " "), " (%)")) +
    theme(
      axis.title = element_text(size = 12),
      axis.text = element_text(size = 8),
      axis.text.x = element_text(hjust = 1, angle = 45),
      strip.placement = "outside",
      strip.background = element_rect(colour = NA),
      legend.position = "none")
})

cowplot::plot_grid(plotlist = gg_qc_seq,
                   nrow = 1,
                   ncol = 3,
                   align = "hv",
                   axis = "trbl")

## Mapping QC

Next, we will check the quality of the mapping step performed by `spaceranger 1.0.0` across libraries. To do so, we will compare the percentage of reads mapped to the genome, and within these mapped reads, the amount of reads mapped to intergenic regions, intronic and exonic regions. We aim to obtain libraries with a high percentage of confidently mapped reads, and specially a high percentage of exonic reads, which correspond with gene expression or RNAs. The reads mapping to intergenic regions suggest contamination of ambient DNA, whereas reads mapping to intronic regions may come from pre-mRNAs or mature spliced isoforms that retain certain introns.

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, verbose = FALSE)
qc_map_vars <- c("Reads_Mapped_Confidently_to_Genome",
                 "Reads_Mapped_Confidently_to_Intergenic_Regions",
                 "Reads_Mapped_Confidently_to_Intronic_Regions",
                 "Reads_Mapped_Confidently_to_Exonic_Regions")

gg_qc_map <- purrr::map(qc_map_vars, function(var) {
  ggplot(qcmetric_df, aes_string(x = "sample_id", y = var, fill = "sample_id")) +
    geom_col() +
    theme_bw() +
    scale_fill_brewer(palette = "Set2") +
    ylim(0, 1) +
    labs(x = "Libraries (Sample IDs)",
         y = stringr::str_c(stringr::str_replace_all(var, "_", " "), " (%)")) +
    theme(
      axis.title = element_text(size = 12),
          axis.text = element_text(size = 10),
          axis.text.x = element_text(hjust = 1, angle = 45),
          strip.placement = "outside",
          strip.background = element_rect(colour = NA),
          legend.position = "none")
})
cowplot::plot_grid(plotlist = gg_qc_map,
                   nrow = 2,
                   ncol = 2,
                   align = "hv",
                   axis = "trbl")

## Sequencing saturation and depth

After assessing mapped reads, it is important to test which is the sequencing saturation and depth for each library. The sequencing saturation is dependent on the library complexity and sequencing depth. The library complexity is the total number of different transcripts present in the library and it varies between the cell types/tissues, whereas the sequencing depth is the number of paired reads per cell. For this reason, we will plot the number of detected genes as a function of depth (sequenced reads). As sequencing depth increases, more genes are detected, but this function reaches a plateau, whereby more sequenced reads does not result in more detected genes; therefore, at this point we assure we sequenced until saturation. More specifically, the sequencing saturation the fraction of confidently mapped, valid cell-barcode, valid UMI reads that had a non-unique (cell-barcode, UMI, gene).

In [ ]:
gg_lib_size <- qcmetric_df %>%
  mutate(Number_of_Reads_mil = Number_of_Reads / 1000000) %>%
  ggplot(aes(x = sample_id, y = Number_of_Reads_mil, fill = sample_id)) +
  geom_bar(stat = "identity") +
  theme_bw() +
  scale_fill_brewer(palette = "Set2") +
  labs(x = "Libraries (SAMPLE IDs)", y = "Library size (in millions)") +
  theme(
    axis.title = element_text(size = 12),
    axis.text = element_text(size = 8),
    axis.text.x = element_text(hjust = 1, angle = 45),
    strip.placement = "outside",
    strip.background = element_rect(colour = NA),
    legend.position = "none")

gg_qc_seq_sat <- qcmetric_df %>%
  mutate(Sequencing_Saturation_perc = Sequencing_Saturation,
         Mean_Reads_per_Spot_tho = Mean_Reads_per_Spot / 1000) %>%
  ggplot(aes(x = Mean_Reads_per_Spot_tho,
             y = Sequencing_Saturation_perc, color = sample_id)) +
  geom_point() +
  theme_bw() +
  scale_color_brewer(palette = "Set2") +
  ylim(0, 1) +
  ggrepel::geom_text_repel(ggplot2::aes(label = sample_id), size = 4) +
  labs(x = "Mean Reads per Spot (in thousands)", y = "Sequencing Saturation") +
  theme(
    axis.title = element_text(size = 12),
    axis.text = element_text(size = 10),
    legend.position = "none")

gg_qc_seq_depth_cell <- qcmetric_df %>%
  mutate(Mean_Reads_per_Spot_tho = Mean_Reads_per_Spot / 1000) %>%
  ggplot(aes(x = Mean_Reads_per_Spot_tho,
             y = Median_Genes_per_Spot, color = sample_id)) +
  geom_point() +
  theme_bw() +
  scale_color_brewer(palette = "Set2") +
  ggrepel::geom_text_repel(ggplot2::aes(label = sample_id), size = 4) +
  labs(x = "Mean Reads per Spot (in thousands)",
                y = "Mean Detected Genes per Spot") +
  theme(
    axis.title = ggplot2::element_text(size = 12),
    axis.text = ggplot2::element_text(size = 10),
    legend.position = "none")

gg_qc_seq_depth <- qcmetric_df %>%
  mutate(Number_of_Reads_mil = Number_of_Reads / 1000000) %>%
  ggplot(aes(x = Number_of_Reads_mil,
             y = Total_Genes_Detected, color = sample_id)) +
  geom_point() +
  theme_bw() +
  scale_color_brewer(palette = "Set2") +
  ggrepel::geom_text_repel(aes(label = sample_id), size = 4) +
  labs(x = "Number of Reads (in millions)", y = "Total Genes Detected") +
  theme(
    axis.title = element_text(size = 12),
    axis.text = element_text(size = 10),
    legend.position = "none")

cowplot::plot_grid(gg_lib_size, gg_qc_seq_sat, 
                   gg_qc_seq_depth_cell, gg_qc_seq_depth, 
                   nrow = 2,
                   ncol = 2,
                   align = "hv",
                   axis = "trbl")

In [ ]:
sessionInfo()